In [ ]:
import argparse
import os
import time
import torch
import pandas as pd
from src import load_finetuned_model_lens_from_dir
from src.utils import (
    filter_correct_data,
    create_full_AOS_dataset,
    create_aos_sequence_variant,
    build_eap_dataset
)
import src
import src.utils
import importlib
import json
import numpy as np


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = "7"

In [ ]:
model_path = 'outputs/models/eap/circuit-eng_finetune-eng/seed_42/aos_sequence_variants/2025-06-18 17:49:56.819794_tflens_hotel_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20'
dataset_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
filtered_data_path = 'test/eng_debug_filtered.csv'
full_aos_path = 'test/eng_debug_full_aos.csv'
sequence_variants_path = 'test/eng_debug_sequence_variant.csv'
eap_output_path = 'test/eng_debug_eap_dataset.csv'

## Filter Dataset

### Normal Pipeline

In [ ]:
print("Starting create EAP dataset pipeline...")

# === Load Model ===
print("Loading fine-tuned model...")
model = load_finetuned_model_lens_from_dir(model_path)
device = (
	torch.device("mps") if torch.backends.mps.is_available()
	else torch.device("cuda") if torch.cuda.is_available()
	else torch.device("cpu")
)
model.to(device)
model.eval()


In [ ]:
# === Step 1: Filter Correct Predictions ===
print("Reading dataset and filtering correct predictions...")
df = pd.read_csv(dataset_path)
print(f"Dataset loaded from {dataset_path} ({len(df)} rows)")
os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
filtered_df = filter_correct_data(
	model,
	df,
	"original_sentence",
	"original_triplet",
	filter_mode="AOS",
	filter_only_correct=False,
	save_path=filtered_data_path
)
filtered_df[filtered_df['is_match']].to_csv(filtered_data_path, index=False)
print(f"Filtered data saved to {filtered_data_path} ({len(filtered_df)} rows)")

In [ ]:
df

In [ ]:
debug_view = filtered_df[~filtered_df['is_match']].copy()

In [ ]:
filtered_df

### Check all models

In [ ]:
# List all files in a directory recursively, but stop at the last folder before a file
import os
def list_files_recursively(directory):
	file_list = []
	for root, dirs, files in os.walk(directory):
		for file in files:
			file_list.append(os.path.join(root, file))
	return file_list
files = list_files_recursively('outputs/models/eap')
models = [os.path.dirname(f) for f in files]
models = list(set(models))
models

In [ ]:
temp_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
df_temp = pd.read_csv(temp_path)
df_temp['original_sentence'] = df_temp['original_sentence'].apply(lambda x: f" {x}")
df_temp.to_csv(temp_path.replace('.csv', '_spaceprefix.csv'), index=False)

In [ ]:
filtered_dfs = {}
for model_path in models:
	model = load_finetuned_model_lens_from_dir(model_path)
	device = (
		torch.device("mps") if torch.backends.mps.is_available()
		else torch.device("cuda") if torch.cuda.is_available()
		else torch.device("cpu")
	)
	model.to(device)
	model.eval()

	print(model_path)

	if 'indo' in model_path:
		dataset_path = 'hotel_dataset/counterfacts/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
		language = 'indo'
	elif 'eng' in model_path:
		dataset_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
		language = 'eng'
	elif 'sunda' in model_path:
		dataset_path = 'hotel_dataset/counterfacts/franken_hotel-su_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
		language = 'sunda'
	else:
		raise ValueError("Unknown model language in path: " + model_path)

	seed = model_path.split('/')[4]
	df = pd.read_csv(dataset_path)
	filtered_data_path = f'temp/{language}_{seed}.csv'
	os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
	id = filtered_data_path.split('/')[-1]
	filtered_df = filter_correct_data(
		model,
		df,
		"original_sentence",
		"original_triplet",
		filter_mode="AOS",
		filter_only_correct=True,
		save_path=filtered_data_path
	)
	filtered_dfs[id] = filtered_df.copy()

In [ ]:
filtered_dfs.keys()

In [ ]:
indexes = set()
first = True
for df in filtered_dfs.values():
	if first:
		indexes = set(df['index'].tolist())
		first = False
	else:
		# Get the intersection of indexes
		indexes.intersection_update(df['index'].tolist())

In [ ]:
len(indexes)

In [ ]:
# dataset_path = 'hotel_dataset/counterfacts/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
# language = 'indo'
# dataset_path = 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
# language = 'eng'
dataset_path = 'hotel_dataset/counterfacts/franken_hotel-su_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
language = 'sunda'

df = pd.read_csv(dataset_path)

# Only take df with index same as indexes
filtered_df = df[df['index'].isin(indexes)].copy()
filtered_df.to_csv(dataset_path.replace('.csv', f'_filtered.csv'), index=False)

## Create Full AOS Dataset

In [ ]:
# === Step 2: Create Full AOS Dataset ===
print("Creating full AOS dataset...")
os.makedirs(os.path.dirname(full_aos_path), exist_ok=True)
full_aos_df = create_full_AOS_dataset(filtered_data_path)
full_aos_df.to_csv(full_aos_path, index=False)
print(f"Full AOS dataset saved to {full_aos_path} ({len(full_aos_df)} rows)")

## Create Sequence Variants

In [ ]:
# === Step 3: Create AOS Sequence Variants ===
print("Creating AOS sequence variants...")
os.makedirs(os.path.dirname(sequence_variants_path), exist_ok=True)
sequence_df = create_aos_sequence_variant(full_aos_path)
sequence_df.to_csv(sequence_variants_path, index=False)
print(f"AOS sequence variants saved to {sequence_variants_path} ({len(sequence_df)} rows)")

In [ ]:
sequence_df

## Build EAP Dataset

In [ ]:
import argparse
import os
import time
import torch
import pandas as pd
from src import load_finetuned_model_lens_from_dir
from src.utils import (
    filter_correct_data,
    create_full_AOS_dataset,
    create_aos_sequence_variant,
    build_eap_dataset
)
import src
import src.utils
import importlib

importlib.reload(src)
importlib.reload(src.utils)

In [ ]:
# === Step 4: Build EAP Dataset ===
print("Building EAP dataset...")
os.makedirs(os.path.dirname(eap_output_path), exist_ok=True)
eap_df = src.utils.build_eap_dataset(
	model=model,
	df=sequence_df,
	sentence_col="original_sentence",
	triplet_col="original_triplet",
	corrupted_col="counterfact3_aspect_replaced",
	corrupted_triplet_col="counterfact_triplet3_aspect_replaced",
	filer_same_length_counterfactuals=True,
	suffix="[A] [O] [S]"
)
eap_df.to_csv(eap_output_path, index=False)
print(f"EAP dataset saved to {eap_output_path} ({len(eap_df)} rows)")

## Fix Counterfact For Entire Input-Triplet pair

### Get Intersection Between Models

In [ ]:
import re

def parse_aos_triplet(triplet_str):
    """
    Parse AOS triplet string to extract Aspect, Opinion, and Sentiment.
    Works with any length of text between the tags.
    
    Args:
        triplet_str: String like "[A] service [O] very good [S] positive"
    
    Returns:
        tuple: (aspect, opinion, sentiment) or None if parsing fails
    """
    # Remove extra whitespaces and strip
    triplet_str = ' '.join(triplet_str.split()).strip()
    
    # Pattern to match [A] content [O] content [S] content
    pattern = r'\[A\]\s*(.*?)\s*\[O\]\s*(.*?)\s*\[S\]\s*(.*?)(?:\s*\[|$)'
    
    match = re.search(pattern, triplet_str)
    
    if match:
        aspect = match.group(1).strip()
        opinion = match.group(2).strip()
        sentiment = match.group(3).strip()
        return (aspect, opinion, sentiment)
    else:
        return None

def parse_multiple_triplets(triplet_str, separator='[SSEP]'):
    """
    Parse multiple AOS triplets separated by a delimiter.
    
    Args:
        triplet_str: String with multiple triplets like "[A] service [O] good [S] positive [SSEP] [A] place [O] nice [S] positive"
        separator: Separator between triplets (default: '[SSEP]')
    
    Returns:
        list: List of (aspect, opinion, sentiment) tuples
    """
    # Split by separator and parse each triplet
    triplets_str = triplet_str.split(separator)
    triplets = []
    
    for t in triplets_str:
        t = t.strip()
        if t == '':
            return [] 
            
        parsed = parse_aos_triplet(t)
        if parsed:
            triplets.append(parsed)
        else:
            print(f"Warning: Could not parse triplet: '{t}'")
    
    return triplets

In [ ]:
counterfact_data_dict = {
    'index': [],
    'original_sentence': [],
    'original_triplet': []
}

lang = 'sunda'
dataset_train_path = f'hotel_dataset/{lang}/no_duplicates/hotel_aste_train_augmented_noreasoning.json'
dataset_dev_test_path = f'hotel_dataset/{lang}/no_duplicates/hotel_aste_dev_test_augmented.json'
with open(dataset_train_path, 'r', encoding='utf-8') as f:
	dataset_train = json.load(f)
with open(dataset_dev_test_path, 'r', encoding='utf-8') as f:
	dataset_dev_test = json.load(f)

for i in range(0, len(dataset_dev_test), 5):
	counterfact_data_dict['index'].append(dataset_dev_test[i]['sentence_id'])
	counterfact_data_dict['original_sentence'].append(dataset_dev_test[i]['input'].replace(' [A] [O] [S]', ''))
	counterfact_data_dict['original_triplet'].append(parse_multiple_triplets(dataset_dev_test[i]['target']))

In [ ]:
valid_indexes = []
for idx, triplets in enumerate(counterfact_data_dict['original_triplet']):
	if len(triplets) == 1:
		triplet = triplets[0]
		aspect, opinion, sentiment = triplet
		if aspect == 'null' or aspect == '':
			print(f"Skipping index {idx} with triplet: {triplet} (aspect is 'null')")
			continue
		if opinion == 'null' or opinion == '':
			print(f"Skipping index {idx} with triplet: {triplet} (opinion is 'null')")
			continue
		valid_indexes.append(idx)

In [ ]:
valid_init_counterfacts = {
    'index': [counterfact_data_dict['index'][i] for i in valid_indexes],
	'original_sentence': [counterfact_data_dict['original_sentence'][i] for i in valid_indexes],
	'original_triplet': [str(counterfact_data_dict['original_triplet'][i]) for i in valid_indexes]
}

In [ ]:
pd.DataFrame(valid_init_counterfacts)

In [ ]:
# List all files in a directory recursively, but stop at the last folder before a file
import os
def list_files_recursively(directory):
	file_list = []
	for root, dirs, files in os.walk(directory):
		for file in files:
			file_list.append(os.path.join(root, file))
	return file_list
files = list_files_recursively('outputs/models/eap')
models = [os.path.dirname(f) for f in files]
models = list(set(models))
models

In [ ]:
filtered_dfs = {}
for model_path in models:
	if f"circuit-{lang}" not in model_path:
		print(f"Skipping model {model_path} as it does not match the language {lang}")
		continue
	model = load_finetuned_model_lens_from_dir(model_path)
	device = (
		torch.device("mps") if torch.backends.mps.is_available()
		else torch.device("cuda") if torch.cuda.is_available()
		else torch.device("cpu")
	)
	model.to(device)
	model.eval()

	print(model_path)

	seed = model_path.split('/')[4]
	filtered_data_path = f'temp/{lang}_{seed}.csv'
	df = pd.DataFrame(valid_init_counterfacts)
	os.makedirs(os.path.dirname(filtered_data_path), exist_ok=True)
	id = filtered_data_path.split('/')[-1]
	filtered_df = filter_correct_data(
		model,
		df,
		"original_sentence",
		"original_triplet",
		filter_mode="AOS",
		max_tokens=100,
		filter_only_correct=True,
		save_path=filtered_data_path
	)
	filtered_dfs[id] = filtered_df.copy()

In [ ]:
filtered_dfs['eng_seed_31415.csv']

In [ ]:
filtered_dfs = {}
results_path = os.listdir('temp')
for path in results_path:
    filtered_dfs[path] = pd.read_csv(os.path.join('temp', path))
filtered_dfs.keys()

In [ ]:
indexes = set()
first = True
for df in filtered_dfs.values():
	if first:
		indexes = set(df['index'].tolist())
		first = False
	else:
		# Get the intersection of indexes
		indexes.intersection_update(df['index'].tolist())

# Convert to list and sort
indexes = sorted(list(indexes))
len(indexes)

In [ ]:
print(indexes)

In [ ]:

for lang in ['indo', 'eng', 'sunda']:
	counterfact_data_dict = {
		'index': [],
		'original_sentence': [],
		'original_triplet': []
	}

	dataset_train_path = f'hotel_dataset/{lang}/no_duplicates/hotel_aste_train_augmented_noreasoning.json'
	dataset_dev_test_path = f'hotel_dataset/{lang}/no_duplicates/hotel_aste_dev_test_augmented.json'
	with open(dataset_train_path, 'r', encoding='utf-8') as f:
		dataset_train = json.load(f)
	with open(dataset_dev_test_path, 'r', encoding='utf-8') as f:
		dataset_dev_test = json.load(f)

	for i in range(0, len(dataset_dev_test), 5):
		if dataset_dev_test[i]['sentence_id'] not in indexes:
			continue
		counterfact_data_dict['index'].append(dataset_dev_test[i]['sentence_id'])
		counterfact_data_dict['original_sentence'].append(dataset_dev_test[i]['input'])
		counterfact_data_dict['original_triplet'].append(dataset_dev_test[i]['target'])

	df_counterfact = pd.DataFrame(counterfact_data_dict)
	df_counterfact['original_pair'] = df_counterfact.apply(lambda row: f"{row['original_sentence']} {row['original_triplet']}", axis=1)
	df_counterfact['corrupted_pair'] = np.nan

	df_counterfact[['index', 'original_pair', 'corrupted_pair']].to_csv(f'hotel_dataset/empty_counterfacts/{lang}_counterfacts.csv', index=False)

### Debug Length Tokens

In [ ]:
dataset_paths = {
	'indo': 'hotel_dataset/counterfacts/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv',
	'eng': 'hotel_dataset/counterfacts/tflens_hotel-en_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv',
	'sunda': 'hotel_dataset/counterfacts/franken_hotel-su_aste_train_augmented_noreasoning_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual.csv'
}

In [ ]:
for lang, dataset_path in dataset_paths.items():
	df_full_aos = create_full_AOS_dataset(dataset_path)
	break

In [ ]:
df_full_aos

In [ ]:
a = model.to_str_tokens("tidak ada sabun di dalam kamar . [A] [O] [S]")
b = model.to_str_tokens("sedia fisika di dalam kamar . [A] [O] [S]")
c = model.to_str_tokens(" tidak ada sabun di dalam kamar . [A] [O] [S]")
d = model.to_str_tokens(" sedia fisika di dalam kamar . [A] [O] [S]")
print('length:', len(a), '|', a)
print('length:', len(b), '|', b)
print('length:', len(c), '|', c)
print('length:', len(d), '|', d)


In [ ]:
a = model.to_str_tokens("[A] sabun [O] tidak ada [S] negative")
b = model.to_str_tokens("[A] fisika [O] sedia [S] positive")
print('length:', len(a), '|', a)
print('length:', len(b), '|', b)